# Same question, GPT-5.6 Luna — via the ChatGPT subscription (OmniRoute)

Third twin of `real-llm.ipynb` / `deepseek-flash.ipynb`. The model is `gpt-5.6-luna`, reached through **OmniRoute** (`localhost:20128/v1`), which wraps the ChatGPT **Pro subscription** (Codex OAuth) as an OpenAI-compatible API.

What the subscription route gives you — probed live 2026-09-24:

| knob the trick needs | llama.cpp | DeepSeek API | **Luna via subscription** |
|---|---|---|---|
| probabilities of the next token | `n_probs` + `post_sampling_probs` | `top_logprobs` ≤ 20 | **none** — `logprobs` accepted, returned `null` |
| `logit_bias` | applied | ignored | ignored |
| exactly one token (`n_predict`/`max_tokens=1`) | yes | yes | **no** — reasoning tokens still spent (31–57 seen) |
| raw prompt control | full | chat only | chat only + a hidden ~2k-token Codex system prompt |
| `gpt-6-luna` | — | — | **refused**: *not supported when using Codex with a ChatGPT account* |

So the one-token trick is **impossible** here: there is no distribution to read. What we *can* do is **estimate** it — ask the same question N times at temperature 1 and count. Same output shape (`{option: probability}` + confidence), different physics:

| | read logprobs (bonsai / DeepSeek) | sample & count (Luna) |
|---|---|---|
| calls per decision | 1 | N (20 here) |
| precision | exact | ± standard error √(p(1−p)/N) |
| off-schema answers | masked / missing | **counted** as `other` — your "mass outside the options" signal, for free |

In [1]:
import collections, concurrent.futures as cf, json, math, os, re, sys, time, urllib.request, urllib.error
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) == "learn" else HERE
sys.path.insert(0, ROOT)
import minijev  # loads ROOT/.env (OMNIROUTE_URL / OMNIROUTE_API_KEY); reused for _marks, confidence and the bonsai comparison

OMNI_URL = os.environ.get("OMNIROUTE_URL", "http://127.0.0.1:20128/v1")
OMNI_KEY = os.environ["OMNIROUTE_API_KEY"]
MODEL = "codex/gpt-5.6-luna"

def omni(path, body=None):
    req = urllib.request.Request(OMNI_URL + path, json.dumps(body).encode() if body else None,
                                 {"Content-Type": "application/json", "Authorization": "Bearer " + OMNI_KEY})
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode()[:400]}") from None

t0 = time.perf_counter()
luna = [m["id"] for m in omni("/models")["data"] if m["id"].startswith("codex/") and "luna" in m["id"]]
print(f"OmniRoute OK in {(time.perf_counter() - t0) * 1000:.0f} ms; luna models: {sorted(set(luna))}")

OmniRoute OK in 280 ms; luna models: ['codex/gpt-5.6-luna', 'codex/gpt-5.6-luna-high', 'codex/gpt-5.6-luna-low', 'codex/gpt-5.6-luna-max', 'codex/gpt-5.6-luna-medium', 'codex/gpt-5.6-luna-xhigh']


## Proof — the three knobs are gone

One request with everything the trick uses: `logprobs`, `top_logprobs`, `max_tokens=1`, `logit_bias` (+100 on an arbitrary id). Note `"stream": False` — OmniRoute streams (SSE) by default. Watch three things in the output: `logprobs` is `null`, `completion_tokens` > 1 (reasoning), and `prompt_tokens` ≈ 2k for a ~70-token question (the hidden Codex system prompt).

In [2]:
Q = ("Text:\nSorry about the outage -- we have reset everyone's limits for the day.\n\nQuestion: How urgent is this?\n"
     "Options:\n1. ignore -- not relevant\n2. today -- act today\n3. now -- stop what you are doing\n"
     "Reply with exactly one character: 1, 2, 3.")
out = omni("/chat/completions", {"model": MODEL, "messages": [{"role": "user", "content": Q}], "stream": False,
                                 "logprobs": True, "top_logprobs": 20, "max_tokens": 1, "logit_bias": {"16": 100}})
c = out["choices"][0]
print("content :", repr(c["message"]["content"]))
print("logprobs:", c.get("logprobs"))
print("usage   :", out["usage"])

content : '2'
logprobs: None
usage   : {'prompt_tokens': 2081, 'completion_tokens': 41, 'total_tokens': 2122, 'completion_tokens_details': {'reasoning_tokens': 34}}


## Step 0 — input dict

Identical to the other two notebooks.

In [3]:
INPUT = {
    "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
    "questions": {
        "is_quota_reset": {"type": "noul", "instructions": "Does this announce a quota reset?"},
        "urgency": {"type": "choice", "instructions": "How urgent is this?",
                    "criteria": {"ignore": "not relevant", "today": "act today",
                                 "now": "stop what you are doing"}},
    },
}
state = INPUT["state"]
print(json.dumps(INPUT, indent=2))

{
  "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
  "questions": {
    "is_quota_reset": {
      "type": "noul",
      "instructions": "Does this announce a quota reset?"
    },
    "urgency": {
      "type": "choice",
      "instructions": "How urgent is this?",
      "criteria": {
        "ignore": "not relevant",
        "today": "act today",
        "now": "stop what you are doing"
      }
    }
  }
}


## Step 1 — sample N times, count

Same prompt text as `minijev`. `reasoning_effort: "none"` so the model answers instead of deliberating (cheaper, faster, and closer to "first-token belief"); `temperature: 1.0` so repeated calls actually differ. Each reply is parsed to its first word; anything that isn't one of our labels is counted as **`other`** rather than dropped — that's the off-schema mass.

Estimate: $\hat p_i = \text{count}_i / N$, standard error $\sqrt{\hat p_i(1-\hat p_i)/N}$. With N = 20 a 0.5 estimate is ±0.11 — coarse, and that is the honest price of having no logprobs. And a count of 0 is **not** a probability of 0: the formula gives ±0.00 there, but the 95% upper bound for 0 hits in N tries is ≈ 3/N (the *rule of three*) — 0.15 at N = 20.

In [4]:
N = 20

def ask_once(body):
    out = omni("/chat/completions", {"model": MODEL, "messages": [{"role": "user", "content": body}],
                                     "stream": False, "reasoning_effort": "none", "temperature": 1.0})
    return out["choices"][0]["message"]["content"] or ""

def first_word(text):
    m = re.match(r"[\s*\"'`(]*([A-Za-z0-9]+)", text)
    return m.group(1) if m else ""

def sample(body, labels, n=N):
    """labels: {name: reply text}. Returns (probabilities incl. 'other', raw counts, replies)."""
    with cf.ThreadPoolExecutor(8) as ex:
        replies = list(ex.map(ask_once, [body] * n))
    by_text = {tok.casefold(): name for name, tok in labels.items()}
    counts = collections.Counter(by_text.get(first_word(r).casefold(), "other") for r in replies)
    probs = {name: counts.get(name, 0) / n for name in list(labels) + ["other"]}
    return probs, counts, replies

def se(p, n=N):
    return math.sqrt(p * (1 - p) / n)

q = INPUT["questions"]["urgency"]
keys = list(q["criteria"])
marks = minijev._marks(len(keys))
menu = "\n".join(f"{m}. {k} -- {q['criteria'][k]}" for m, k in zip(marks, keys))
body = (f"Text:\n{state}\n\nQuestion: {q['instructions']}\nOptions:\n{menu}\n"
        f"Reply with exactly one character: {', '.join(marks)}.")

t0 = time.perf_counter()
probs, counts, replies = sample(body, dict(zip(keys, marks)))
print(f"{N} calls in {(time.perf_counter() - t0) * 1000:.0f} ms (8 in parallel)")
print("raw replies:", collections.Counter(replies))
for k, p in probs.items():
    print(f"   {k:7} p={p:.2f} ± {se(p):.2f}")
in_schema = {k: v for k, v in probs.items() if k != "other"}
print("pick:", max(in_schema, key=in_schema.get), "| confidence (margin):", minijev.confidence(in_schema),
      "| off-schema share:", probs["other"])

20 calls in 6229 ms (8 in parallel)
raw replies: Counter({'2': 18, '1': 2})
   ignore  p=0.10 ± 0.07
   today   p=0.90 ± 0.07
   now     p=0.00 ± 0.00
   other   p=0.00 ± 0.00
pick: today | confidence (margin): 0.8 | off-schema share: 0.0


## Step 2 — the one-liners, side by side with the local bonsai

`noul` / `choice` / `score` rebuilt on `sample()`, with `minijev`'s exact body text. Luna numbers are counts out of N, so they move in steps of 1/N and wobble run to run; bonsai's are exact logprobs.

In [5]:
def noul(state, question, criteria=None):
    crit = f"\ntrue means: {criteria['true']}\nfalse means: {criteria['false']}" if criteria else ""
    body = f"Text:\n{state}\n\nQuestion: {question}{crit}\nReply with exactly one word: Yes or No."
    p, _, _ = sample(body, {"true": "Yes", "false": "No"})
    return p["true"], minijev.confidence({k: v for k, v in p.items() if k != "other"}), p["other"]

def choice(state, question, options):
    keys = list(options)
    marks = minijev._marks(len(keys))
    menu = "\n".join(f"{m}. {k} -- {options[k]}" for m, k in zip(marks, keys))
    body = (f"Text:\n{state}\n\nQuestion: {question}\nOptions:\n{menu}\n"
            f"Reply with exactly one character: {', '.join(marks)}.")
    p, _, _ = sample(body, dict(zip(keys, marks)))
    real = {k: v for k, v in p.items() if k != "other"}
    return real, minijev.confidence(real), p["other"]

def score(state, question, levels):
    marks = minijev._marks(len(levels))
    menu = "\n".join(f"{m}. {d}" for m, d in zip(marks, levels))
    body = (f"Text:\n{state}\n\nQuestion: {question}\nScale:\n{menu}\n"
            f"Reply with exactly one character: {', '.join(marks)}.")
    p, _, _ = sample(body, {str(i + 1): m for i, m in enumerate(marks)})
    real = {k: v for k, v in p.items() if k != "other"}
    z = sum(real.values()) or 1.0
    return sum(int(k) * v for k, v in real.items()) / z, real, minijev.confidence(real), p["other"]

LEVELS = ["no impact", "minor inconvenience", "noticeable disruption", "serious outage", "total outage"]
SCORE_Q = "How severe was the incident being apologised for?"

t0 = time.perf_counter()
L = {"noul": noul(state, INPUT["questions"]["is_quota_reset"]["instructions"]),
     "choice": choice(state, q["instructions"], q["criteria"]),
     "score": score(state, SCORE_Q, LEVELS)}
print(f"luna: 3 decisions x {N} samples in {(time.perf_counter() - t0) * 1000:.0f} ms")

try:
    B = {"noul": minijev.noul(state, INPUT["questions"]["is_quota_reset"]["instructions"]),
         "choice": minijev.choice(state, q["instructions"], q["criteria"]),
         "score": minijev.score(state, SCORE_Q, LEVELS)}
except Exception as e:
    B = None
    print("local bonsai unavailable:", e)

r2 = lambda d: {k: round(v, 2) for k, v in d.items()}
print("\nnoul P(true)")
print("   gpt-5.6-luna :", L["noul"][0], "| confidence", L["noul"][1], "| off-schema", L["noul"][2])
if B: print("   bonsai-2-27b :", round(B["noul"][0], 4), "| confidence", B["noul"][1])
print("\nchoice")
print("   gpt-5.6-luna :", max(L["choice"][0], key=L["choice"][0].get), r2(L["choice"][0]), "| confidence", L["choice"][1], "| off-schema", L["choice"][2])
if B: print("   bonsai-2-27b :", max(B["choice"][0], key=B["choice"][0].get), r2(B["choice"][0]), "| confidence", B["choice"][1])
print("\nscore E[level]")
print("   gpt-5.6-luna :", round(L["score"][0], 2), r2(L["score"][1]), "| confidence", L["score"][2], "| off-schema", L["score"][3])
if B: print("   bonsai-2-27b :", round(B["score"][0], 3), r2(B["score"][1]), "| confidence", B["score"][2])

luna: 3 decisions x 20 samples in 16862 ms



noul P(true)
   gpt-5.6-luna : 1.0 | confidence 1.0 | off-schema 0.0
   bonsai-2-27b : 0.9839 | confidence 0.9678

choice
   gpt-5.6-luna : today {'ignore': 0.1, 'today': 0.9, 'now': 0.0} | confidence 0.8 | off-schema 0.0
   bonsai-2-27b : today {'ignore': 0.26, 'today': 0.62, 'now': 0.12} | confidence 0.3627

score E[level]
   gpt-5.6-luna : 4.0 {'1': 0.0, '2': 0.0, '3': 0.0, '4': 1.0, '5': 0.0} | confidence 1.0 | off-schema 0.0
   bonsai-2-27b : 3.638 {'1': 0.02, '2': 0.08, '3': 0.16, '4': 0.7, '5': 0.03} | confidence 0.5477


## When to use which

| need | use |
|---|---|
| calibrated probabilities, 1 call, hostile input | llama.cpp + `minijev` (logit_bias + logprobs) |
| hosted, cheap, probabilities from one call | DeepSeek (`top_logprobs`, no bias) |
| a frontier model's *answer*, uncertainty only roughly | Luna via subscription: sample N, count; N× the calls, ±√(p(1−p)/N) error |

Subscription caveats: every call burns ~2k prompt tokens of the **shared Pro quota** (the hidden Codex system prompt), and OmniRoute warns that proxying a product OAuth session may get the account restricted — keep N small and the volume light.